# BabyLM 2026 Semantic-Cloze Multi-Task Experiment Walkthrough

This notebook is a study guide for the clean multi-task BabyLM experiment. It shows the inputs, commands, recorded outputs, and interpretation step by step.

The cells are written in a simple code-block style. The outputs shown here are recorded from the completed run so the notebook can be reviewed without rerunning the full GPU experiment.

## 1. Repository and experiment paths

First we define the main paths used throughout the experiment.

In [1]:
# Description: Define the repository paths used by the walkthrough.
from pathlib import Path

REPO = Path('/home/paperspace/BabyLM-2026-submission')
EXP = REPO / 'experiments/multitask_ewok_semantic_cloze_v1'
CONFIG = EXP / 'configs/multitask_ewok_semantic_cloze_v1.yaml'
SEMANTIC_CLOZE = EXP / 'data/semantic_cloze_ranking.jsonl'

print('Repo exists:', REPO.exists())
print('Experiment exists:', EXP.exists())
print('Config exists:', CONFIG.exists())
print('Semantic cloze data exists:', SEMANTIC_CLOZE.exists())

Repo exists: True
Experiment exists: True
Config exists: True
Semantic cloze data exists: True


## 2. Experiment configuration

The selected run trains a compact BERT-style masked language model. The important design constraint is that the final model remains compatible with the official BabyLM `mlm` evaluation backend.

In [2]:
# Description: Load the YAML configuration and inspect the compact BERT model shape.
import yaml

cfg = yaml.safe_load(CONFIG.read_text())
cfg['model']

{'hidden_size': 256,
 'num_hidden_layers': 4,
 'num_attention_heads': 4,
 'intermediate_size': 1024,
 'max_position_embeddings': 512}

In [3]:
# Description: Inspect the training hyperparameters, including early stopping.
cfg['training']

{'max_length': 128,
 'batch_size': 96,
 'max_steps': 10000,
 'calibration_mlm_steps': 2000,
 'learning_rate': 0.0005,
 'weight_decay': 0.01,
 'warmup_fraction': 0.06,
 'max_grad_norm': 1.0,
 'eval_every': 500,
 'eval_batches': 40,
 'precision': 'fp32',
 'early_stopping': {'enabled': True,
  'monitor': 'val_mlm_loss',
  'patience_evals': 2,
  'min_delta': 0.01,
  'start_after_step': 9000}}

## 3. Multi-task mixture

The model is trained mostly as an MLM, with small auxiliary objectives added. The semantic-cloze ranking objective is only 5 percent of the sampling mixture.

In [4]:
# Description: Extract the sampling weights for each task in the multi-task mixture.
task_weights = {
    name: spec['probability']
    for name, spec in cfg['tasks'].items()
}
task_weights

{'mlm': 0.6,
 'rtd': 0.1,
 'connective': 0.075,
 'definiteness': 0.075,
 'collocation': 0.05,
 'grammar_minpair': 0.05,
 'semantic_cloze_ranking': 0.05}

## 4. Semantic-cloze input examples

The new idea was to add small conceptual plausibility supervision using the MLM head. Each row has:

- a prompt with exactly one `[MASK]`
- a plausible single-token completion called `good`
- an implausible single-token completion called `bad`
- a domain label for inspection

In [5]:
# Description: Load the first semantic-cloze ranking examples used as auxiliary supervision.
import json

examples = [json.loads(line) for line in SEMANTIC_CLOZE.read_text().splitlines()[:3]]
examples

[{'task': 'semantic_cloze_ranking',
  'prompt': 'In everyday life, a shovel is used for [MASK].',
  'good': 'digging',
  'bad': 'reading',
  'domain': 'affordance'},
 {'task': 'semantic_cloze_ranking',
  'prompt': 'In everyday life, the purpose of a spoon is [MASK].',
  'good': 'eating',
  'bad': 'driving',
  'domain': 'affordance'},
 {'task': 'semantic_cloze_ranking',
  'prompt': 'People know that a broom is used for [MASK].',
  'good': 'sweeping',
  'bad': 'reading',
  'domain': 'affordance'}]

In [6]:
# Description: Count how many semantic-cloze examples are in the generated dataset.
num_semantic_cloze_examples = sum(1 for _ in SEMANTIC_CLOZE.open())
num_semantic_cloze_examples

1000

## 5. Semantic-cloze loss

The ranking loss uses the MLM head directly. This matters because it avoids adding a classifier head that the official BabyLM submission pipeline would not use.

In [7]:
# Description: Show the semantic-cloze pairwise ranking loss in plain terms.
print('score_good = log p(good token at [MASK] | prompt)')
print('score_bad  = log p(bad token at [MASK] | prompt)')
print('loss       = softplus(-(score_good - score_bad))')
print()
print('The loss is small when the model assigns higher probability to the good completion.')

score_good = log p(good token at [MASK] | prompt)
score_bad  = log p(bad token at [MASK] | prompt)
loss       = softplus(-(score_good - score_bad))

The loss is small when the model assigns higher probability to the good completion.


## 6. Generate or inspect auxiliary data

The branch already stores the generated auxiliary data used by this run. The semantic-cloze file can be regenerated with the experiment-specific script.

In [8]:
# Description: Show the command used to regenerate the semantic-cloze examples.
print('python experiments/multitask_ewok_semantic_cloze_v1/scripts/generate_semantic_cloze.py')

python experiments/multitask_ewok_semantic_cloze_v1/scripts/generate_semantic_cloze.py


## 7. GPU preflight

Before training, verify that CUDA is visible. The experiment was run on GPU with `CUDA_VISIBLE_DEVICES=0`.

In [9]:
# Description: Show the GPU preflight command and the recorded CUDA result.
print('CUDA_VISIBLE_DEVICES=0 python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"')
print('\nRecorded output:')
print('True')
print('NVIDIA GPU visible as device 0')

CUDA_VISIBLE_DEVICES=0 python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"

Recorded output:
True
NVIDIA GPU visible as device 0


## 8. Training command

This is the command for the selected run. It trains with the mixture above, evaluates every 500 steps, and uses early stopping after step 9000.

In [10]:
# Description: Show the exact GPU training command for the selected run.
print('CUDA_VISIBLE_DEVICES=0 python experiments/multitask_distributional_bert/scripts/train_multitask_bert.py \\')
print('  --config experiments/multitask_ewok_semantic_cloze_v1/configs/multitask_ewok_semantic_cloze_v1.yaml \\')
print('  --seed 1')

CUDA_VISIBLE_DEVICES=0 python experiments/multitask_distributional_bert/scripts/train_multitask_bert.py \
  --config experiments/multitask_ewok_semantic_cloze_v1/configs/multitask_ewok_semantic_cloze_v1.yaml \
  --seed 1


## 9. Training outputs

The key training question was: does early stopping work, and which checkpoint should we keep? The run stopped at step 9500. The best validation MLM loss was reached at step 5000.

In [11]:
# Description: Load the recorded number of batches sampled from each task.
counts_path = REPO / 'experiments/run_records/multitask_ewok_semantic_cloze_v1/task_counts.json'
task_counts = json.loads(counts_path.read_text())
task_counts

{'mlm': 6307,
 'rtd': 800,
 'connective': 588,
 'definiteness': 620,
 'collocation': 435,
 'grammar_minpair': 362,
 'semantic_cloze_ranking': 388}

In [12]:
# Description: Display the key validation checkpoints and early stopping decision.
from IPython.display import Markdown, display

display(Markdown('''| step | task | val_mlm_loss | best_val_mlm_loss | bad_eval_count | early_stop |
|---:|---|---:|---:|---:|---|
| 5000 | definiteness | 6.8105 | 6.8105 | 0 | False |
| 8500 | mlm | 6.9363 | 6.8105 | 0 | False |
| 9000 | mlm | 6.8363 | 6.8105 | 1 | False |
| 9500 | mlm | 6.8733 | 6.8105 | 2 | True |'''))

| step | task | val_mlm_loss | best_val_mlm_loss | bad_eval_count | early_stop |
|---:|---|---:|---:|---:|---|
| 5000 | definiteness | 6.8105 | 6.8105 | 0 | False |
| 8500 | mlm | 6.9363 | 6.8105 | 0 | False |
| 9000 | mlm | 6.8363 | 6.8105 | 1 | False |
| 9500 | mlm | 6.8733 | 6.8105 | 2 | True |

Interpretation: early stopping was active, but intentionally did not start counting until after step 9000. At step 9000 there was one non-improving eval, and at step 9500 there were two. Since `patience_evals = 2`, training stopped at 9500.

## 10. Fast local evaluation

The fast eval compared the plain MLM baseline with the semantic-cloze multi-task checkpoints. The important result is mixed: BLiMP dropped, but EWoK, entity tracking, and reading metrics improved over the local MLM baseline.

In [13]:
# Description: Show the official fast-eval command used for the uploaded model.
print('bash experiments/multitask_distributional_bert/scripts/run_official_fast_eval.sh \\')
print('  alonsopg/babylm-2026-semantic-cloze-strict-small \\')
print('  mlm \\')
print('  strict-small')

bash experiments/multitask_distributional_bert/scripts/run_official_fast_eval.sh \
  alonsopg/babylm-2026-semantic-cloze-strict-small \
  mlm \
  strict-small


In [14]:
# Description: Display the local fast-eval comparison table.
display(Markdown('''| System | BLiMP | BLiMP Supplement | EWoK | Entity Tracking | Reading Eye | Reading SPR |
|---|---:|---:|---:|---:|---:|---:|
| normal_bert_mlm_final | 64.33 | 54.00 | 50.55 | 15.56 | 2.47 | 2.42 |
| multitask_ewok_semantic_cloze_v1_best | 54.10 | 57.60 | 50.73 | 36.61 | 7.56 | 3.11 |
| multitask_ewok_semantic_cloze_v1_final | 55.26 | 54.40 | 52.00 | 27.44 | 8.18 | 3.71 |'''))

| System | BLiMP | BLiMP Supplement | EWoK | Entity Tracking | Reading Eye | Reading SPR |
|---|---:|---:|---:|---:|---:|---:|
| normal_bert_mlm_final | 64.33 | 54.00 | 50.55 | 15.56 | 2.47 | 2.42 |
| multitask_ewok_semantic_cloze_v1_best | 54.10 | 57.60 | 50.73 | 36.61 | 7.56 | 3.11 |
| multitask_ewok_semantic_cloze_v1_final | 55.26 | 54.40 | 52.00 | 27.44 | 8.18 | 3.71 |

## 11. Official full evaluation summary

The final model was uploaded to Hugging Face and evaluated through the official BabyLM scripts where possible.

In [15]:
# Description: Display the official zero-shot full-evaluation summary.
display(Markdown('''| Section | Score |
|---|---:|
| BLiMP filtered | 55.35 |
| BLiMP supplement filtered | 53.29 |
| EWoK filtered | 50.45 |
| Entity tracking | 27.55 |
| COMPS | 50.63 |
| Reading eye-tracking | 8.18 |
| Reading SPR | 3.71 |'''))

| Section | Score |
|---|---:|
| BLiMP filtered | 55.35 |
| BLiMP supplement filtered | 53.29 |
| EWoK filtered | 50.45 |
| Entity tracking | 27.55 |
| COMPS | 50.63 |
| Reading eye-tracking | 8.18 |
| Reading SPR | 3.71 |

In [16]:
# Description: Display the official finetuning evaluation summary.
display(Markdown('''| Finetuning task | Main scores |
|---|---|
| BoolQ | acc 0.6679, F1 0.7755, MCC 0.2061 |
| MNLI | acc 0.4269 |
| MRPC | acc 0.6912, F1 0.8131, MCC 0.1300 |
| MultiRC | acc 0.5858, F1 0.2132, MCC 0.0859 |
| QQP | acc 0.7044, F1 0.5816, MCC 0.3554 |
| RTE | acc 0.5468, F1 0.5191, MCC 0.0910 |
| WSC | acc 0.5769, F1 0.1538, MCC -0.0381 |'''))

| Finetuning task | Main scores |
|---|---|
| BoolQ | acc 0.6679, F1 0.7755, MCC 0.2061 |
| MNLI | acc 0.4269 |
| MRPC | acc 0.6912, F1 0.8131, MCC 0.1300 |
| MultiRC | acc 0.5858, F1 0.2132, MCC 0.0859 |
| QQP | acc 0.7044, F1 0.5816, MCC 0.3554 |
| RTE | acc 0.5468, F1 0.5191, MCC 0.0910 |
| WSC | acc 0.5769, F1 0.1538, MCC -0.0381 |

## 12. Minimal submission bundle

The minimal submission uses the final Hugging Face model and the collated official-eval artifact.

In [17]:
# Description: Show the final Hugging Face model and minimal submission artifact paths.
print('Model: alonsopg/babylm-2026-semantic-cloze-strict-small')
print('Backend: mlm')
print('Track: strict-small')
print('Artifact: shared_task_submission/semantic_cloze/artifacts/all_full_preds_and_fast_scores_mlm.json')
print('Bundle: shared_task_submission/semantic_cloze_minimal_submission.zip')

Model: alonsopg/babylm-2026-semantic-cloze-strict-small
Backend: mlm
Track: strict-small
Artifact: shared_task_submission/semantic_cloze/artifacts/all_full_preds_and_fast_scores_mlm.json
Bundle: shared_task_submission/semantic_cloze_minimal_submission.zip


## 13. AoA status

AoA was attempted with the official script, but the script expects Hugging Face checkpoint revisions such as `chck_1M`, `chck_2M`, ..., `chck_100M`. The uploaded model only has the final revision, so those revision downloads returned 404 and no `surprisal.json` was produced.

We did not fake this by copying the final checkpoint to all revision names. The collated artifact therefore leaves AoA as null.

In [18]:
# Description: Show the AoA command and the recorded missing-revision blocker.
print('CUDA_VISIBLE_DEVICES=0 bash scripts/eval_aoa.sh \\')
print('  alonsopg/babylm-2026-semantic-cloze-strict-small \\')
print('  mlm \\')
print('  strict-small \\')
print('  evaluation_data/full_eval/aoa/cdi_childes.json \\')
print('  results')
print('\nRecorded result: missing checkpoint revisions chck_1M through chck_100M; AoA remains null.')

CUDA_VISIBLE_DEVICES=0 bash scripts/eval_aoa.sh \
  alonsopg/babylm-2026-semantic-cloze-strict-small \
  mlm \
  strict-small \
  evaluation_data/full_eval/aoa/cdi_childes.json \
  results

Recorded result: missing checkpoint revisions chck_1M through chck_100M; AoA remains null.


## 14. What the experiment says

The semantic-cloze multi-task idea appears to help the targeted conceptual/plausibility direction modestly, especially in the local fast EWoK result and entity/reading metrics. It does not dominate the plain MLM baseline overall, because BLiMP drops substantially.

So the honest conclusion is: this is a valid minimal multi-task submission and a useful experiment, but not a clearly stronger all-around BabyLM model. The clean story is that small semantic ranking supervision can shift the model toward conceptual plausibility while trading off syntactic acceptability.